In [1]:
import pandas as pd, numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100, "display.width", 200)

# Works whether you run from project root or from notebooks/
DATA = next(p for p in [Path("data/fertilizer.csv"), Path("../data/fertilizer.csv")] if p.exists())
OUT_DIR = DATA.parent / "processed"; OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA, low_memory=False)
print("Full dataset:", df.shape)
print("\nALL COLUMNS:\n", list(df.columns))

# ---- Filter: Wheat + grain (case-insensitive) ----
crop_ok  = df["crop"].astype(str).str.strip().str.lower().eq("wheat")
part_ok  = df["yield_part"].astype(str).str.strip().str.lower().eq("grain")
w = df[crop_ok & part_ok].copy()
print("\nWheat grain records:", len(w))

# ---- Helper: find columns by keyword (names of soil/admin fields vary) ----
def find(*keys):
    return [c for c in w.columns if any(k in c.lower() for k in keys)]

id_cols    = [c for c in ["trial_id", "record_id"] if c in w.columns]
loc_cols   = find("location", "site", "country", "adm", "region", "zone", "woreda")
soil_cols  = find("ph", "soil", "oc", "carbon")
coord_cols = find("lat", "lon")
fert_cols  = [c for c in ["N_fertilizer","P_fertilizer","K_fertilizer","Zn_fertilizer","S_fertilizer"] if c in w.columns]
print("\nID columns:", id_cols)
print("Location/admin columns:", loc_cols)
print("Soil-like columns:", soil_cols)
print("Coordinate columns:", coord_cols)
print("Fertilizer columns:", fert_cols)

# ---- Counts ----
if "trial_id" in w.columns:
    print("\nDistinct trial_id:", w["trial_id"].nunique(), "| missing trial_id:", w["trial_id"].isna().sum())
for c in loc_cols:
    print(f"Distinct {c}: {w[c].nunique()}")

# ---- Records by administrative region ----
for c in find("adm1", "region"):
    print(f"\nRecords by {c}:\n", w[c].value_counts(dropna=False))

# ---- Missing values in key fields ----
key = ["yield"] + soil_cols + coord_cols + fert_cols + ["yield_moisture", "yield_isfresh"]
key = [c for c in dict.fromkeys(key) if c in w.columns]
miss = pd.DataFrame({"missing": w[key].isna().sum(), "pct": (w[key].isna().mean()*100).round(1)})
print("\nMISSING VALUES (key fields):\n", miss.sort_values("pct", ascending=False))

# ---- Yield distribution ----
print("\nYIELD summary:\n", w["yield"].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))
for c in ["yield_moisture", "yield_isfresh", "yield_part"]:
    if c in w.columns:
        print(f"\n{c} value counts:\n", w[c].value_counts(dropna=False).head(15))

# ---- Fertilizer-rate distributions ----
print("\nFERTILIZER rate summary:\n", w[fert_cols].describe().T)
print("\nZero / NaN counts per fertilizer column:")
for c in fert_cols:
    print(f"  {c}: zero={(w[c]==0).sum()}, NaN={w[c].isna().sum()}, >0={(w[c]>0).sum()}")

# ---- Distinct fertilizer-rate combinations ----
combo = (w.groupby(fert_cols, dropna=False).size()
           .reset_index(name="n_rows").sort_values("n_rows", ascending=False))
print("\nDistinct N-P-K-Zn-S combinations:", len(combo))
print(combo.head(20))

# ---- Duplicates ----
print("\nFully duplicated rows:", w.duplicated().sum())
if "record_id" in w.columns:
    print("Duplicated record_id:", w["record_id"].duplicated().sum())
dup_key = [c for c in ["trial_id","season","variety"] + loc_cols[:1] + fert_cols + ["yield"] if c in w.columns]
print("Rows sharing identical trial/season/variety/location/fertilizer/yield:",
      w.duplicated(subset=dup_key, keep=False).sum())

# ---- Save (nothing dropped yet) ----
out = OUT_DIR / "ethiosoil_wheat_grain_trials.csv"
w.to_csv(out, index=False)
print("\nSaved:", out, w.shape)

Full dataset: (4316, 50)

ALL COLUMNS:
 ['trial_id', 'season', 'treatment', 'crop', 'variety', 'variety_type', 'previous_crop', 'yield_part', 'yield', 'yield_moisture', 'yield_isfresh', 'uncertainty', 'uncertainty_type', 'row_spacing', 'plant_density', 'dataset_id', 'record_id', 'reference', 'on_farm', 'is_survey', 'country', 'adm1', 'location', 'longitude', 'latitude', 'geo_from_source', 'planting_date', 'harvest_date', 'fertilizer_type', 'N_fertilizer', 'N_splits', 'P_fertilizer', 'K_fertilizer', 'Zn_fertilizer', 'S_fertilizer', 'N_organic', 'P_organic', 'K_organic', 'OM_used', 'OM_type', 'OM_amount', 'inoculated', 'irrigated', 'land_prep_method', 'soil_type', 'soil_pH', 'soil_sand', 'soil_clay', 'soil_SOC', 'soil_P']

Wheat grain records: 1535

ID columns: ['trial_id', 'record_id']
Location/admin columns: ['country', 'adm1', 'location']
Soil-like columns: ['location', 'inoculated', 'soil_type', 'soil_pH', 'soil_sand', 'soil_clay', 'soil_SOC', 'soil_P']
Coordinate columns: ['longitud

In [2]:
import pandas as pd, numpy as np, io
from pathlib import Path
from contextlib import redirect_stdout

pd.set_option("display.max_columns", 100, "display.width", 200, "display.max_rows", 200)

P = next(p for p in [Path("data/processed/ethiosoil_wheat_grain_trials.csv"),
                     Path("../data/processed/ethiosoil_wheat_grain_trials.csv")] if p.exists())
w = pd.read_csv(P, low_memory=False)

fert = ["N_fertilizer","P_fertilizer","K_fertilizer","Zn_fertilizer","S_fertilizer"]
soil = ["soil_pH","soil_SOC","soil_P","soil_sand","soil_clay","soil_type"]
other = ["yield_moisture","yield_isfresh","uncertainty","latitude","longitude","adm1",
         "planting_date","harvest_date","variety","season","previous_crop","irrigated",
         "N_splits","fertilizer_type","OM_used","N_organic","P_organic","K_organic",
         "plant_density","row_spacing","on_farm","is_survey"]

buf = io.StringIO()
with redirect_stdout(buf):
    print("=== MISSING VALUES ===")
    cols = [c for c in ["yield"]+soil+fert+other if c in w.columns]
    print(pd.DataFrame({"missing": w[cols].isna().sum(),
                        "pct": (w[cols].isna().mean()*100).round(1)}).sort_values("pct", ascending=False))

    print("\n=== YIELD ===")
    print(w["yield"].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))
    for c in ["yield_moisture","yield_isfresh","uncertainty_type","yield_part"]:
        print(f"\n{c}:\n", w[c].value_counts(dropna=False).head(10))

    print("\n=== FERTILIZER RATES ===")
    print(w[fert].describe().T)
    for c in fert:
        print(f"{c}: zero={(w[c]==0).sum()}, NaN={w[c].isna().sum()}, >0={(w[c]>0).sum()}")
    combo = w.groupby(fert, dropna=False).size().reset_index(name="n_rows").sort_values("n_rows", ascending=False)
    print("\nDistinct N-P-K-Zn-S combinations:", len(combo))
    print(combo.head(15))

    print("\n=== REGIONS ===")
    print(w.groupby("adm1", dropna=False).agg(rows=("yield","size"),
          trials=("trial_id","nunique"), locations=("location","nunique")))

    print("\n=== TRIALS (rows, locations, seasons, treatments, varieties) ===")
    print(w.groupby("trial_id").agg(rows=("yield","size"), locations=("location","nunique"),
          seasons=("season","nunique"), treatments=("treatment","nunique"),
          varieties=("variety","nunique"), adm1=("adm1","first"),
          soil_pH_missing=("soil_pH", lambda s: s.isna().mean().round(2))).sort_values("rows", ascending=False))

    print("\n=== DUPLICATES ===")
    print("Exact duplicate rows (ignoring record_id):", w.drop(columns=["record_id"]).duplicated().sum())
    key = ["trial_id","season","variety","location"] + fert + ["yield"]
    g = w[w.duplicated(subset=key, keep=False)]
    print("Rows in 'same key' groups:", len(g), "| number of groups:", g.groupby(key, dropna=False).ngroups)
    sizes = g.groupby(key, dropna=False).size()
    print("Group size distribution:\n", sizes.value_counts().sort_index().head(10))
    # which columns differ inside these groups?
    diff = {c: int((g.groupby(key, dropna=False)[c].nunique(dropna=False) > 1).sum())
            for c in w.columns if c not in key}
    print("\nColumns that differ inside duplicate groups (count of groups):")
    print({k: v for k, v in sorted(diff.items(), key=lambda x: -x[1]) if v > 0})
    ex = g.sort_values(key).head(8)
    print("\nExample rows:\n", ex[["trial_id","record_id","treatment","season","variety","location","yield","plant_density","planting_date"]])

report = buf.getvalue()
(P.parent / "step1b_audit_report.txt").write_text(report)
print(report)

=== MISSING VALUES ===
                 missing    pct
yield_isfresh       1535  100.0
yield_moisture      1535  100.0
row_spacing         1535  100.0
P_organic           1535  100.0
K_organic           1535  100.0
N_organic           1535  100.0
plant_density       1535  100.0
previous_crop       1301   84.8
soil_sand           1205   78.5
soil_clay            977   63.6
fertilizer_type      946   61.6
soil_P               868   56.5
N_splits             807   52.6
soil_pH              587   38.2
soil_SOC             554   36.1
soil_type            410   26.7
uncertainty          377   24.6
season               214   13.9
variety              187   12.2
harvest_date         170   11.1
planting_date        170   11.1
irrigated             80    5.2
adm1                  80    5.2
yield                  0    0.0
K_fertilizer           0    0.0
N_fertilizer           0    0.0
longitude              0    0.0
latitude               0    0.0
S_fertilizer           0    0.0
Zn_fertilizer    